## LangChain 활용해보기

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w4" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : d:\gangsa\hanwha-agent


In [2]:
# 함수 3개 

def build_prompt(question: str) -> str:
    return f"[ 규정 질문 ]: {question}"

def call_llm(prompt: str) -> str:
    return '{"answer": "출장 일비는 1일 3만원입니다.", "doc_id": "DOC-HR-012"}'

def parser(text: str) -> dict:
    import json
    return json.loads(text)


In [ ]:
from langchain_core.runnables import Runnable, RunnableLambda

step = RunnableLambda(build_prompt)
print(type(step))
# step.invoke()


<class 'langchain_core.runnables.base.RunnableLambda'>


In [ ]:

pipeline = RunnableLambda(build_prompt) | call_llm | parser
print(type(pipeline))
# pipeline.invoke()


<class 'langchain_core.runnables.base.RunnableSequence'>


In [ ]:
def lookup(payload: dict) -> str:
    # payload : {"doc_id": "...."} 조회할 문서id dict 형태로 받기 
    known = {"DOC-HR-012", "DOC-PU-007", "DOC-SE-002"}
    if payload["doc_id"] not in known:
        raise ValueError("모르는 문서...")
    return f"{payload['doc_id']} 조회 완료"

lookup_chain = RunnableLambda(lookup)

# 여러개 요청 
inputs = [
    {"doc_id": "DOC-HR-012"},
    {"doc_id": "DOC-XX-999"}, # 없는 doc_id
    {"doc_id": "DOC-PU-007"},
    {"doc_id": "DOC-SE-002"},
]

# try:
#     # lookup_chain.invoke({"doc_id": "DOC-HR-012"})  한개 
#     lookup_chain.batch(inputs) # 여러개 -> 한건이 죽으면 전체가 다 죽는다. 
# except ValueError as e:
#     print("실패.....", e)

results = lookup_chain.batch(inputs, return_exceptions=True)
for one in results:
    print(one)


DOC-HR-012 조회 완료
모르는 문서...
DOC-PU-007 조회 완료
DOC-SE-002 조회 완료


In [10]:
from langchain_core.language_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser

llm = FakeListChatModel(responses=["부산 출장 비용은 1일 2만원입니다."])

out = llm.invoke("부산 출장 일비는?")
print(out)
print(type(out))

text_chain = llm | StrOutputParser()
print(text_chain.invoke("부산 출장 일비는?"))


content='부산 출장 비용은 1일 2만원입니다.' additional_kwargs={} response_metadata={} id='lc_run--01a0c7f5-de1b-7f22-bb35-439f37174995-0' tool_calls=[] invalid_tool_calls=[]
<class 'langchain_core.messages.ai.AIMessage'>
부산 출장 비용은 1일 2만원입니다.


In [14]:
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

ANSWER = "일비 2만원 - 숙박 실비"

gllm = GenericFakeChatModel(messages=iter([AIMessage(content=ANSWER)]))
pieces = list((gllm | StrOutputParser()).stream("부산 출장 일비는?"))

print("조각 개수 : ", len(pieces))
print(pieces)


gllm2 = GenericFakeChatModel(messages=iter([AIMessage(content=ANSWER)]))
print("이어서 출력: ", end="") # 줄내림 없이 옆으로 출력 
for piece in ((gllm2 | StrOutputParser()).stream("부산 출장 일비는?")): 
    print(piece, end="", flush=True)

print()



조각 개수 :  9
['일비', ' ', '2만원', ' ', '-', ' ', '숙박', ' ', '실비']
이어서 출력: 일비 2만원 - 숙박 실비


In [17]:
from langchain_core.runnables import RunnablePassthrough

enrich = RunnablePassthrough.assign(
    doc_id=lambda d: "DOC-HR-011", 
    grade=lambda d: "일반"
)

print(enrich.invoke({"question": "부산 출장 일비는?"}))
print(type(enrich))

line = enrich | RunnableLambda(lambda d: f"[{d['doc_id']}/{d['grade']}] {d['question']}")
print(line.invoke({"question": "부산 출장 일비는?"}))


{'question': '부산 출장 일비는?', 'doc_id': 'DOC-HR-011', 'grade': '일반'}
<class 'langchain_core.runnables.passthrough.RunnableAssign'>
[DOC-HR-011/일반] 부산 출장 일비는?


In [21]:
from langchain_core.runnables import RunnableParallel

both = RunnableParallel(
    upper=RunnableLambda(lambda s: s.upper()), 
    length=RunnableLambda(lambda s: len(s))
)
print("parrallel : ", both.invoke("DOC-hr-002"))

auto = RunnableLambda(lambda s: s) | {
    "upper": RunnableLambda(lambda s: s.upper()), 
    "length": RunnableLambda(lambda s: len(s))
}
print("dict : ", auto.invoke("DOC-hr-002"))


parrallel :  {'upper': 'DOC-HR-002', 'length': 10}
dict :  {'upper': 'DOC-HR-002', 'length': 10}
